# 05 — Ablation Studies (5-Step Sequential)

Each step varies one axis, holds everything else constant.
Results from each step carry forward as the fixed value for subsequent steps.

1. **Step 1 — Input Representation:** Feature layers (L0–L3), spatial/temporal, graph structure
2. **Step 2 — Encoder Architecture:** ST-GCN vs Transformer vs LSTM vs 1D-CNN
3. **Step 3 — Pre-Training Regime:** Random init vs SimCLR vs SimCLR + Aux (RQ2)
4. **Step 4 — Few-Shot Method:** ProtoNet vs k-NN vs Linear probe
5. **Step 5 — K-Shot Sensitivity:** K = 1, 3, 5, 8, 10, 15

In [ ]:
import subprocess, sys, os, zipfile

# ── Colab setup ──────────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/FineBadminton/baddiev2_colab.zip'
    PROJECT_PATH = '/content/Baddiev2'
    if not os.path.exists(PROJECT_PATH):
        print("Extracting project files...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
        print("Done.")
    os.chdir(os.path.join(PROJECT_PATH, 'notebooks'))
    sys.path.insert(0, PROJECT_PATH)
    print(f"CWD: {os.getcwd()}")
else:
    print("Local run")

sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
from tqdm import tqdm
from src.config import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_DIR = Path('/content/drive/MyDrive/Baddiev2 project/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {device}")

SKELETON_DIR = FB_SKELETONS_GDINO_V2
print(f"Skeleton dir: {SKELETON_DIR}")

## Ablation Runner Helper

Reusable function that takes a config, runs the full pipeline,
and returns results dict.

In [ ]:
from src.data.graph_builder import GraphBuilder
from src.data.dataset import FineBadmintonDataset, EpisodicSampler, collate_episode
from src.models.stgcn_model import STGCN
from src.models.transformer_encoder import SkeletonTransformer
from src.models.lstm_encoder import SkeletonLSTM
from src.models.cnn1d_encoder import SkeletonCNN1D
from src.models.proto_net import PrototypicalNetwork
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression


def build_encoder(cfg, adjacency, device):
    """Build encoder; derives num_nodes from adjacency shape to handle single_player."""
    feature_dim = FEATURE_DIMS[cfg.ablation.feature_layer]
    enc_type = cfg.ablation.encoder
    num_nodes = adjacency.shape[-1]  # 17 or 34 depending on single_player

    if enc_type == 'stgcn':
        return STGCN(
            in_channels=feature_dim,
            num_nodes=num_nodes,
            adjacency=adjacency,
            num_layers=cfg.stgcn.num_layers,
            base_channels=cfg.stgcn.base_channels,
            embedding_dim=cfg.stgcn.embedding_dim,
            temporal_kernel=cfg.stgcn.temporal_kernel,
            dropout=cfg.stgcn.dropout,
        ).to(device)
    elif enc_type == 'transformer':
        return SkeletonTransformer(
            in_channels=feature_dim * num_nodes,  # derived, not hardcoded
            d_model=cfg.transformer.d_model,
            nhead=cfg.transformer.nhead,
            num_layers=cfg.transformer.num_layers,
            embedding_dim=cfg.transformer.embedding_dim,
            dropout=cfg.transformer.dropout,
        ).to(device)
    elif enc_type == 'lstm':
        return SkeletonLSTM(
            in_channels=feature_dim,
            num_nodes=num_nodes,
            hidden_dim=cfg.lstm.hidden_dim,
            num_layers=cfg.lstm.num_layers,
            embedding_dim=cfg.lstm.embedding_dim,
            dropout=cfg.lstm.dropout,
            bidirectional=cfg.lstm.bidirectional,
        ).to(device)
    elif enc_type == 'cnn1d':
        return SkeletonCNN1D(
            in_channels=feature_dim,
            num_nodes=num_nodes,
            channels=cfg.cnn1d.channels,
            kernel_size=cfg.cnn1d.kernel_size,
            embedding_dim=cfg.cnn1d.embedding_dim,
            dropout=cfg.cnn1d.dropout,
        ).to(device)
    else:
        raise ValueError(f"Unknown encoder: {enc_type}")


def extract_embeddings(encoder, dataset, indices, device):
    encoder.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for i in indices:
            x, y = dataset[i]
            x = x.unsqueeze(0).to(device)
            emb = encoder(x).cpu().numpy()[0]
            embeddings.append(emb)
            labels.append(y)
    return np.array(embeddings), np.array(labels)


def train_one_epoch(encoder, dataset, train_idx, optimizer, cfg, device):
    encoder.train()
    train_labels = [dataset.get_labels()[i] for i in train_idx]
    sampler = EpisodicSampler(
        labels=train_labels,
        n_way=cfg.proto.n_way,
        k_shot=cfg.proto.k_shot,
        n_query=cfg.proto.n_query,
        episodes_per_epoch=cfg.proto.episodes_per_epoch,
    )
    epoch_loss, epoch_acc, n_episodes = 0.0, 0.0, 0
    proto_net = PrototypicalNetwork(encoder, distance=cfg.proto.distance)

    for episode_indices in sampler:
        actual_indices = [train_idx[i] for i in episode_indices]
        batch = [dataset[i] for i in actual_indices]
        support_x, support_y, query_x, query_y = collate_episode(
            batch, cfg.proto.n_way, cfg.proto.k_shot, cfg.proto.n_query
        )
        support_x, support_y = support_x.to(device), support_y.to(device)
        query_x, query_y = query_x.to(device), query_y.to(device)

        support_emb = encoder(support_x)
        query_emb = encoder(query_x)
        n_way_actual = int(support_y.unique().numel())
        prototypes = proto_net.compute_prototypes(support_emb, support_y, n_way_actual)
        distances = proto_net.compute_distances(query_emb, prototypes)
        log_probs = torch.nn.functional.log_softmax(-distances, dim=1)
        loss = torch.nn.functional.nll_loss(log_probs, query_y)
        acc = ((-distances).argmax(dim=1) == query_y).float().mean().item()

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc
        n_episodes += 1

    return epoch_loss / n_episodes, epoch_acc / n_episodes


def evaluate_protonet(encoder, dataset, support_idx, test_idx, device):
    support_emb, support_labels = extract_embeddings(encoder, dataset, support_idx, device)
    test_emb, test_labels = extract_embeddings(encoder, dataset, test_idx, device)

    prototypes = {c: support_emb[support_labels == c].mean(axis=0)
                  for c in np.unique(support_labels)}
    preds = np.array([
        min(prototypes, key=lambda c: np.linalg.norm(emb - prototypes[c]))
        for emb in test_emb
    ])
    all_labels = list(range(NUM_CLASSES))
    macro_f1 = f1_score(test_labels, preds, average='macro',
                        labels=all_labels, zero_division=0)
    per_class = f1_score(test_labels, preds, average=None,
                         labels=all_labels, zero_division=0)
    return {'macro_f1': macro_f1, 'per_class_f1': per_class.tolist()}


def run_ablation(cfg, skeleton_dir=None, ssl_path=None, device=device):
    """Full ablation run with 5-fold CV. Returns aggregated results dict."""
    if skeleton_dir is None:
        skeleton_dir = SKELETON_DIR

    graph_builder = GraphBuilder(
        use_inter_player=cfg.ablation.use_inter_player,
        single_player=cfg.ablation.single_player,
    )
    adjacency = graph_builder.build_adjacency().to(device)
    num_nodes = adjacency.shape[-1]  # 17 or 34

    # Single-player transform: slice dataset output to first 17 nodes
    transform = None
    if cfg.ablation.single_player:
        transform = lambda x: x[:, :, :num_nodes]

    dataset = FineBadmintonDataset(
        skeleton_dir=skeleton_dir,
        shot_window=cfg.data.shot_window,
        feature_layer=cfg.ablation.feature_layer,
        transform=transform,
    )
    splits = dataset.get_fold_splits(n_folds=cfg.data.num_folds, seed=cfg.data.random_seed)

    # Load SSL weights if available (only L2 checkpoint exists)
    ssl_checkpoint = None
    if ssl_path is None:
        ssl_path = MODELS_DIR / f'ssl_pretrained_{cfg.ablation.feature_layer}.pt'
    if cfg.ablation.use_pretrained and ssl_path.exists() and cfg.ablation.encoder == 'stgcn':
        ssl_checkpoint = torch.load(ssl_path, map_location=device, weights_only=False)
        print(f"  SSL weights: {ssl_path.name}")
    else:
        print(f"  No SSL weights — random init")

    fold_f1s, fold_per_class = [], []

    for fold_idx, (train_idx, val_idx, test_idx) in enumerate(splits):
        encoder = build_encoder(cfg, adjacency, device)
        if ssl_checkpoint is not None:
            encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])

        optimizer = optim.Adam(
            encoder.parameters(),
            lr=cfg.proto.lr * cfg.proto.encoder_lr_scale,
            weight_decay=1e-5,
        ) if cfg.proto.fine_tune_encoder else None

        if optimizer is None:
            encoder.eval()
            for p in encoder.parameters():
                p.requires_grad = False

        best_val_f1, best_state = 0.0, None
        for epoch in range(cfg.proto.epochs):
            train_one_epoch(encoder, dataset, train_idx, optimizer, cfg, device)
            if (epoch + 1) % 5 == 0:
                val_res = evaluate_protonet(encoder, dataset, train_idx, val_idx, device)
                if val_res['macro_f1'] > best_val_f1:
                    best_val_f1 = val_res['macro_f1']
                    best_state = {k: v.clone() for k, v in encoder.state_dict().items()}

        if best_state is not None:
            encoder.load_state_dict(best_state)

        test_result = evaluate_protonet(encoder, dataset, train_idx, test_idx, device)
        fold_f1s.append(test_result['macro_f1'])
        fold_per_class.append(test_result['per_class_f1'])
        print(f"    Fold {fold_idx+1}: F1={test_result['macro_f1']:.3f}")

    result = {
        'macro_f1_mean': float(np.mean(fold_f1s)),
        'macro_f1_std': float(np.std(fold_f1s)),
        'fold_f1s': [float(f) for f in fold_f1s],
        'per_class_f1_mean': np.mean(fold_per_class, axis=0).tolist(),
    }
    print(f"  → Macro-F1: {result['macro_f1_mean']:.3f} ± {result['macro_f1_std']:.3f}")
    return result

print("Ablation runner ready.")

---
## Step 1a — Feature Engineering (Input Representation)

Fix encoder = ST-GCN, graph = full dual-player, episodic fine-tuning.
SSL weights loaded for L2 only (only checkpoint that exists); others use random init.

**This is the most important ablation** — it answers which signal layers
(raw position → kinematics → court context → joint angles) carry strategy-relevant information.

| Layer | Features | Dim |
|-------|----------|----:|
| L0 | x, y | 2 |
| L1 | + velocity, acceleration | 6 |
| L2 | + dist_to_net, dist_to_center, dist_to_opponent | 9 |
| L3 | + elbow, shoulder, knee angles | 12 |

In [ ]:
step1a_configs = {
    'L0_raw_xy':       get_config('L0', **{'ablation.feature_layer': 'L0', 'ablation.use_pretrained': False}),
    'L1_kinematics':   get_config('L1', **{'ablation.feature_layer': 'L1', 'ablation.use_pretrained': False}),
    'L2_court_ctx':    get_config('L2', **{'ablation.feature_layer': 'L2'}),  # SSL checkpoint exists
    'L3_joint_angles': get_config('L3', **{'ablation.feature_layer': 'L3', 'ablation.use_pretrained': False}),
}

print(f"Step 1a — Feature Engineering ({len(step1a_configs)} variants)")
print(f"{'Variant':<20} {'Layer':<5} {'Dim':>4}  {'Init'}")
print("-" * 45)
for name, cfg in step1a_configs.items():
    fl = cfg.ablation.feature_layer
    ssl = 'SSL' if cfg.ablation.use_pretrained else 'random'
    print(f"  {name:<20} {fl:<5} {FEATURE_DIMS[fl]:>4}  {ssl}")

In [ ]:
step1a_results = {}

for name, cfg in step1a_configs.items():
    print(f"\n{'='*50}")
    print(f"Step 1a — {name} ({cfg.ablation.feature_layer}, {FEATURE_DIMS[cfg.ablation.feature_layer]}-dim)")
    step1a_results[name] = run_ablation(cfg, device=device)

# Summary table
print(f"\n{'='*60}")
print(f"Step 1a Summary — Feature Engineering")
print(f"{'Variant':<22} {'Macro-F1':>10} {'± Std':>8}  Per-class F1")
print("-" * 70)
best_name_1a, best_f1_1a = None, 0
for name, res in step1a_results.items():
    pc = [f"{v:.2f}" for v in res['per_class_f1_mean']]
    print(f"  {name:<20} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}  {pc}")
    if res['macro_f1_mean'] > best_f1_1a:
        best_f1_1a = res['macro_f1_mean']
        best_name_1a = name

print(f"\nBest: {best_name_1a} ({best_f1_1a:.3f})")
BEST_FEATURE_LAYER = step1a_configs[best_name_1a].ablation.feature_layer
print(f"→ Fixing feature_layer = {BEST_FEATURE_LAYER} for Steps 1b, 2, 3, 4, 5")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
names = list(step1a_results.keys())
means = [step1a_results[n]['macro_f1_mean'] for n in names]
stds  = [step1a_results[n]['macro_f1_std']  for n in names]
bars = ax.bar(names, means, yerr=stds, capsize=5,
              color=['#4C72B0','#55A868','#C44E52','#8172B2'])
ax.set_ylabel('Macro-F1')
ax.set_title('Step 1a — Feature Engineering Ablation')
ax.set_ylim(0, 0.8)
ax.axhline(0.2, color='gray', linestyle='--', linewidth=0.8, label='Random baseline (5-class)')
ax.legend()
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{m:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_feature_layers.png', dpi=150)
plt.show()

---
## Step 2 — Encoder Architecture

Fix input = best from Steps 1a + 1b (feature layer + graph structure).
SSL weights only available for ST-GCN/L2; others use random init.

Vary: ST-GCN (graph conv) vs Transformer (self-attention) vs LSTM (recurrent) vs 1D-CNN (temporal conv).

In [ ]:
step2_configs = {
    'stgcn':       get_config('stgcn', **{
        'ablation.feature_layer': BEST_FEATURE_LAYER,
        'ablation.encoder': 'stgcn',
        'ablation.use_inter_player': USE_INTER_PLAYER,
        'ablation.single_player': SINGLE_PLAYER,
    }),
    'transformer': get_config('transformer', **{
        'ablation.feature_layer': BEST_FEATURE_LAYER,
        'ablation.encoder': 'transformer',
        'ablation.use_inter_player': USE_INTER_PLAYER,
        'ablation.single_player': SINGLE_PLAYER,
    }),
    'lstm':        get_config('lstm', **{
        'ablation.feature_layer': BEST_FEATURE_LAYER,
        'ablation.encoder': 'lstm',
        'ablation.use_inter_player': USE_INTER_PLAYER,
        'ablation.single_player': SINGLE_PLAYER,
    }),
    'cnn1d':       get_config('cnn1d', **{
        'ablation.feature_layer': BEST_FEATURE_LAYER,
        'ablation.encoder': 'cnn1d',
        'ablation.use_inter_player': USE_INTER_PLAYER,
        'ablation.single_player': SINGLE_PLAYER,
    }),
}

print(f"Step 2 — Encoder Architecture")
print(f"  Fixed: feature={BEST_FEATURE_LAYER}, graph={BEST_GRAPH}")
for name in step2_configs:
    print(f"  {name}")

step2_results = {}

for name, cfg in step2_configs.items():
    print(f"\n{'='*50}")
    print(f"Step 2 — {name}")
    step2_results[name] = run_ablation(cfg, device=device)

# Summary
print(f"\n{'='*60}")
print(f"Step 2 Summary — Encoder Architecture")
print(f"  Fixed: feature={BEST_FEATURE_LAYER}, graph={BEST_GRAPH}")
print(f"{'Encoder':<15} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 35)
best_name_2, best_f1_2 = None, 0
for name, res in step2_results.items():
    print(f"  {name:<13} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}")
    if res['macro_f1_mean'] > best_f1_2:
        best_f1_2 = res['macro_f1_mean']
        best_name_2 = name

print(f"\nBest: {best_name_2} ({best_f1_2:.3f})")
BEST_ENCODER = best_name_2
print(f"→ Fixing encoder = {BEST_ENCODER} for Steps 3–5")

---
## Step 2 — Encoder Architecture

Fix input = best from Step 1, episodic ProtoNet fine-tuning.
SSL weights only available for ST-GCN; others use random init.

Vary: ST-GCN vs Transformer vs LSTM vs 1D-CNN.

In [ ]:
step2_configs = {
    'stgcn': get_config('stgcn', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': 'stgcn',
    }),
    'transformer': get_config('transformer', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': 'transformer',
    }),
    'lstm': get_config('lstm', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': 'lstm',
    }),
    'cnn1d': get_config('cnn1d', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': 'cnn1d',
    }),
}

print(f"Step 2 variants (feature_layer={BEST_STEP1}):")
for name in step2_configs:
    print(f"  {name}")

In [ ]:
step2_results = {}

for name, cfg in step2_configs.items():
    print(f"\n{'='*50}")
    print(f"Step 2 — {name}")
    step2_results[name] = run_ablation(cfg, device=device)

# Summary
print(f"\n{'='*60}")
print(f"Step 2 Summary — Encoder Architecture (feature_layer={BEST_STEP1})")
print(f"{'Encoder':<15} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 35)
best_name, best_f1 = None, 0
for name, res in step2_results.items():
    print(f"{name:<15} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}")
    if res['macro_f1_mean'] > best_f1:
        best_f1 = res['macro_f1_mean']
        best_name = name
print(f"\nBest: {best_name} ({best_f1:.3f})")

BEST_STEP2 = best_name
print(f"→ Fixing encoder = {BEST_STEP2} for Step 3")

---
## Step 3 — Pre-Training Regime (RQ2)

Fix encoder + input = best from Steps 1-2.
Vary: random init vs SSL pre-trained (SimCLR + Aux).

Note: SimCLR-only (no aux) would require a separate SSL training run.
If that checkpoint doesn't exist, we skip it.

In [ ]:
step3_configs = {
    'random_init': get_config('random', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': BEST_STEP2,
        'ablation.use_pretrained': False,
    }),
    'ssl_plus_aux': get_config('ssl_aux', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': BEST_STEP2,
        'ablation.use_pretrained': True,
    }),
}

# Only add contrastive-only if a separate checkpoint exists
ssl_cl_only_path = MODELS_DIR / f'ssl_pretrained_{BEST_STEP1}_cl_only.pt'
if ssl_cl_only_path.exists():
    step3_configs['ssl_contrastive_only'] = get_config('ssl_only', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': BEST_STEP2,
        'ablation.use_pretrained': True,
    })
    print(f"Found contrastive-only checkpoint: {ssl_cl_only_path.name}")
else:
    print(f"No contrastive-only checkpoint — skipping that variant")

print(f"\nStep 3 variants (feature={BEST_STEP1}, encoder={BEST_STEP2}):")
for name in step3_configs:
    print(f"  {name}")

In [ ]:
step3_results = {}

for name, cfg in step3_configs.items():
    print(f"\n{'='*50}")
    print(f"Step 3 — {name}")
    # For contrastive-only, point to the separate checkpoint
    ssl_override = ssl_cl_only_path if name == 'ssl_contrastive_only' else None
    step3_results[name] = run_ablation(cfg, ssl_path=ssl_override, device=device)

# Summary
print(f"\n{'='*60}")
print(f"Step 3 Summary — Pre-Training Regime (RQ2)")
print(f"{'Variant':<25} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 45)
best_name, best_f1 = None, 0
for name, res in step3_results.items():
    print(f"{name:<25} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}")
    if res['macro_f1_mean'] > best_f1:
        best_f1 = res['macro_f1_mean']
        best_name = name
print(f"\nBest: {best_name} ({best_f1:.3f})")

BEST_STEP3_PRETRAINED = step3_configs[best_name].ablation.use_pretrained
print(f"→ Fixing use_pretrained = {BEST_STEP3_PRETRAINED} for Step 4")

---
## Step 4 — Few-Shot Method

Fix encoder + input + pre-training = best from Steps 1-3.
Train encoder with episodic ProtoNet, then freeze and compare classifiers on the embeddings.

Vary: ProtoNet vs k-NN (k=3,5) vs Linear probe.

In [ ]:
step4_methods = {
    'protonet': {'method': 'protonet'},
    'knn_3': {'method': 'knn', 'knn_k': 3},
    'knn_5': {'method': 'knn', 'knn_k': 5},
    'linear_probe': {'method': 'linear_probe'},
}

print(f"Step 4 — Classifier comparison")
print(f"Using best config: feature={BEST_STEP1}, encoder={BEST_STEP2}, pretrained={BEST_STEP3_PRETRAINED}")
for name in step4_methods:
    print(f"  {name}")

In [ ]:
# Train the best encoder once, extract embeddings, then compare classifiers
best_cfg = get_config('step4_best', **{
    'ablation.feature_layer': BEST_STEP1,
    'ablation.encoder': BEST_STEP2,
    'ablation.use_pretrained': BEST_STEP3_PRETRAINED,
})

# Build graph and dataset
graph_builder = GraphBuilder(
    use_inter_player=best_cfg.ablation.use_inter_player,
    single_player=best_cfg.ablation.single_player,
)
adjacency = graph_builder.build_adjacency().to(device)

dataset = FineBadmintonDataset(
    skeleton_dir=SKELETON_DIR,
    shot_window=best_cfg.data.shot_window,
    feature_layer=BEST_STEP1,
)
splits = dataset.get_fold_splits(n_folds=best_cfg.data.num_folds, seed=best_cfg.data.random_seed)

# Load SSL if available
ssl_path = MODELS_DIR / f'ssl_pretrained_{BEST_STEP1}.pt'
ssl_checkpoint = None
if BEST_STEP3_PRETRAINED and ssl_path.exists() and BEST_STEP2 == 'stgcn':
    ssl_checkpoint = torch.load(ssl_path, map_location=device, weights_only=False)
    print(f"SSL weights: {ssl_path.name}")

# Train encoder per fold, extract embeddings, evaluate all classifiers
step4_results = {name: [] for name in step4_methods}

for fold_idx, (train_idx, val_idx, test_idx) in enumerate(splits):
    encoder = build_encoder(best_cfg, adjacency, device)
    if ssl_checkpoint is not None:
        encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])

    optimizer = optim.Adam(
        encoder.parameters(),
        lr=best_cfg.proto.lr * best_cfg.proto.encoder_lr_scale,
        weight_decay=1e-5,
    )

    # Episodic training
    best_val_f1, best_state = 0.0, None
    for epoch in range(best_cfg.proto.epochs):
        train_one_epoch(encoder, dataset, train_idx, optimizer, best_cfg, device)
        if (epoch + 1) % 5 == 0:
            val_res = evaluate_protonet(encoder, dataset, train_idx, val_idx, device)
            if val_res['macro_f1'] > best_val_f1:
                best_val_f1 = val_res['macro_f1']
                best_state = {k: v.clone() for k, v in encoder.state_dict().items()}

    if best_state is not None:
        encoder.load_state_dict(best_state)

    # Extract embeddings with frozen encoder
    all_emb, all_labels = extract_embeddings(
        encoder, dataset, list(range(len(dataset))), device
    )

    # Evaluate each classifier
    for name, method_cfg in step4_methods.items():
        X_train, y_train = all_emb[train_idx], all_labels[train_idx]
        X_test, y_test = all_emb[test_idx], all_labels[test_idx]
        all_labels_list = list(range(NUM_CLASSES))

        if method_cfg['method'] == 'protonet':
            prototypes = {}
            for c in np.unique(y_train):
                prototypes[c] = X_train[y_train == c].mean(axis=0)
            preds = np.array([
                min(prototypes, key=lambda c: np.linalg.norm(x - prototypes[c]))
                for x in X_test
            ])
        elif method_cfg['method'] == 'knn':
            clf = KNeighborsClassifier(n_neighbors=method_cfg['knn_k'])
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)
        elif method_cfg['method'] == 'linear_probe':
            clf = LogisticRegression(max_iter=1000, C=1.0)
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)

        f1 = f1_score(y_test, preds, average='macro',
                       labels=all_labels_list, zero_division=0)
        step4_results[name].append(f1)

    print(f"  Fold {fold_idx+1} done")

# Summary
print(f"\n{'='*60}")
print(f"Step 4 Summary — Few-Shot Method")
print(f"{'Method':<15} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 35)
best_name, best_f1 = None, 0
for name, f1s in step4_results.items():
    mean_f1, std_f1 = np.mean(f1s), np.std(f1s)
    print(f"{name:<15} {mean_f1:>10.3f} {std_f1:>8.3f}")
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_name = name

# Convert to standard format
step4_results = {
    name: {
        'macro_f1_mean': float(np.mean(f1s)),
        'macro_f1_std': float(np.std(f1s)),
        'fold_f1s': [float(f) for f in f1s],
    }
    for name, f1s in step4_results.items()
}

BEST_STEP4 = best_name
print(f"\nBest: {BEST_STEP4} ({best_f1:.3f})")

all_results = {
    'best_config': {
        'feature_layer': BEST_FEATURE_LAYER,
        'graph': BEST_GRAPH,
        'use_inter_player': USE_INTER_PLAYER,
        'single_player': SINGLE_PLAYER,
        'encoder': BEST_ENCODER,
        'use_pretrained': BEST_STEP3_PRETRAINED,
        'classifier': BEST_STEP4,
    },
    'step1a_feature_layer': step1a_results,
    'step1b_graph_structure': step1b_results,
    'step2_encoder': step2_results,
    'step3_pretraining': step3_results,
    'step4_classifier': step4_results,
    'step5_kshot': {str(k): v for k, v in step5_results.items()},
}

with open(RESULTS_DIR / 'ablation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"Saved to {RESULTS_DIR / 'ablation_results.json'}")

print(f"\n{'='*60}")
print(f"ABLATION SUMMARY")
print(f"{'='*60}")
print(f"Best feature layer:  {BEST_FEATURE_LAYER}  (Step 1a)")
print(f"Best graph:          {BEST_GRAPH}  (Step 1b)")
print(f"Best encoder:        {BEST_ENCODER}  (Step 2)")
print(f"Best pre-training:   {'SSL+Aux' if BEST_STEP3_PRETRAINED else 'Random init'}  (Step 3)")
print(f"Best classifier:     {BEST_STEP4}  (Step 4)")
print(f"{'='*60}")

In [ ]:
k_values = [1, 3, 5, 8, 10]
step5_results = {}

for k in k_values:
    # Adjust n_query so k + n_query ≤ 11 (move_to_net min in train fold)
    n_query = min(5, max(1, 11 - k))
    print(f"\n{'='*50}")
    print(f"Step 5 — K={k} (n_query={n_query})")

    cfg = get_config(f'kshot_{k}', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': BEST_STEP2,
        'ablation.use_pretrained': BEST_STEP3_PRETRAINED,
        'proto.k_shot': k,
        'proto.n_query': n_query,
    })
    step5_results[k] = run_ablation(cfg, device=device)

# Summary
print(f"\n{'='*60}")
print(f"Step 5 Summary — K-Shot Sensitivity")
print(f"{'K':<5} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 25)
for k, res in step5_results.items():
    print(f"{k:<5} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ks = list(step5_results.keys())
means = [step5_results[k]['macro_f1_mean'] for k in ks]
stds = [step5_results[k]['macro_f1_std'] for k in ks]
ax.errorbar(ks, means, yerr=stds, marker='o', capsize=5, linewidth=2)
ax.set_xlabel('K (support shots per class)')
ax.set_ylabel('Macro-F1')
ax.set_title('K-Shot Sensitivity')
ax.set_xticks(ks)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_kshot_curve.png', dpi=150)
plt.show()

---
## Save All Results

In [ ]:
all_results = {
    'best_config': {
        'feature_layer': BEST_STEP1,
        'encoder': BEST_STEP2,
        'use_pretrained': BEST_STEP3_PRETRAINED,
        'classifier': BEST_STEP4,
    },
    'step1_input': step1_results,
    'step2_encoder': step2_results,
    'step3_pretraining': step3_results,
    'step4_classifier': step4_results,
    'step5_kshot': {str(k): v for k, v in step5_results.items()},
}

with open(RESULTS_DIR / 'ablation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"Results saved to {RESULTS_DIR / 'ablation_results.json'}")

# Print grand summary
print(f"\n{'='*60}")
print(f"ABLATION SUMMARY")
print(f"{'='*60}")
print(f"Best feature layer:  {BEST_STEP1}")
print(f"Best encoder:        {BEST_STEP2}")
print(f"Best pre-training:   {'SSL+Aux' if BEST_STEP3_PRETRAINED else 'Random init'}")
print(f"Best classifier:     {BEST_STEP4}")
print(f"{'='*60}")